# Manually label recipe assistant results

Open this notebook from the folder containing `results.json`. Run the cells in order, then click **Good** or **Bad** for each response. Each click immediately saves to `manual_labels.json`; rerunning the notebook resumes at the first unlabeled response.

The notebook kernel needs `ipywidgets`. If it is not installed, run `%pip install ipywidgets` in a separate notebook cell, restart the kernel, and run these cells again. The notebook does not install packages or call an API.

In [13]:
import hashlib
import json
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display

BASE_DIR = Path.cwd()
RESULTS_PATH = BASE_DIR / "results.json"
LABELS_PATH = BASE_DIR / "manual_labels.json"

if not RESULTS_PATH.is_file():
    raise FileNotFoundError(f"Missing {RESULTS_PATH}. Open the notebook from the folder containing results.json.")

try:
    results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
except json.JSONDecodeError as exc:
    raise ValueError(f"{RESULTS_PATH.name} is not valid JSON: {exc}") from exc
if not isinstance(results, list):
    raise ValueError("results.json must contain a list of result objects.")
for index, row in enumerate(results, start=1):
    if (not isinstance(row, dict)
            or not isinstance(row.get("question"), str)
            or not row["question"]
            or not isinstance(row.get("output"), str)):
        raise ValueError(f"Result {index} must have a nonempty question and a string output.")

def result_id(row):
    content = json.dumps([row["question"], row["output"]], ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(content.encode("utf-8")).hexdigest()

if LABELS_PATH.exists():
    try:
        saved = json.loads(LABELS_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"{LABELS_PATH.name} is not valid JSON; no labels were overwritten: {exc}") from exc
    if (not isinstance(saved, dict) or saved.get("schema_version") != 1
            or not isinstance(saved.get("labels"), dict)):
        raise ValueError(f"{LABELS_PATH.name} has an unsupported format; no labels were overwritten.")
    labels = saved["labels"]
    if any(not isinstance(entry, dict) or entry.get("label") not in ("good", "bad")
           or not isinstance(entry.get("question"), str) for entry in labels.values()):
        raise ValueError(f"{LABELS_PATH.name} contains invalid labels; no labels were overwritten.")
else:
    labels = {}

print(f"Loaded {len(results)} results and {len(labels)} saved labels.")

Loaded 24 results and 0 saved labels.


In [14]:
good_button = widgets.Button(description="Good", button_style="success")
bad_button = widgets.Button(description="Bad", button_style="danger")
progress = widgets.Label()
feedback = widgets.Label()
response_view = widgets.Output()
current = {"index": None}

def save_labels():
    payload = {"schema_version": 1, "labels": labels}
    temporary_path = LABELS_PATH.with_suffix(".json.tmp")
    temporary_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    temporary_path.replace(LABELS_PATH)

def show_next():
    pending = [index for index, row in enumerate(results) if result_id(row) not in labels]
    labeled_count = len(results) - len(pending)
    progress.value = f"Labeled {labeled_count} of {len(results)} results"
    current["index"] = pending[0] if pending else None
    good_button.disabled = bad_button.disabled = not pending
    with response_view:
        clear_output(wait=True)
        if not pending:
            print("All results are labeled.")
            return
        index = current["index"]
        row = results[index]
        print(f"Result {index + 1} of {len(results)}")
        print(f"Category: {row.get('category', 'unknown')} | Type: {row.get('type', 'unknown')}")
        print(f"\nQuestion:\n{row['question']}")
        print(f"\nAssistant response:\n{row['output']}")

def label_current(label):
    index = current["index"]
    if index is None:
        return
    row = results[index]
    key = result_id(row)
    labels[key] = {"question": row["question"], "label": label}
    try:
        save_labels()
    except OSError as exc:
        del labels[key]
        feedback.value = f"Could not save label: {exc}"
        return
    feedback.value = f"Saved {label} to {LABELS_PATH.name}."
    show_next()

good_button.on_click(lambda _: label_current("good"))
bad_button.on_click(lambda _: label_current("bad"))
display(widgets.VBox([progress, response_view, widgets.HBox([good_button, bad_button]), feedback]))
show_next()